# Séance 11 — Hugging Face : réutiliser des modèles pré-entraînés (vision)

**Objectif.** Séance de synthèse : aucune notion nouvelle à proprement parler, seulement de
l'outillage. On réutilise des modèles pré-entraînés à la fois côté **discriminatif**
(classification d'images, lien avec les séances 7-9) et côté **génératif** (lien avec la
séance 10). Entièrement en vision — pas de NLP.

Quatre parties, de plus en plus impliquées :

1. **Inférence directe**, sans aucun entraînement (`pipeline`).
2. **Sous le capot** : `AutoImageProcessor` + `AutoModel`, pour observer les features
   extraites avant toute tête de classification.
3. **Transfer learning** : backbone gelé + tête entraînée avec le `Trainer` habituel, puis
   dégel partiel des dernières couches — même démarche que le mini-projet CNN (séance 9),
   mais avec un backbone pré-entraîné à la place d'un entraînement from scratch.
4. **Réutiliser un modèle génératif pré-entraîné** (`diffusers`), en écho à la séance 10.


In [ ]:
# !pip install -q torch torchvision transformers datasets diffusers accelerate

import glob

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import matplotlib.pyplot as plt
from PIL import Image

from transformers import AutoImageProcessor, AutoModel, pipeline
from diffusers import DDPMPipeline

from training_toolbox import Trainer, freeze, unfreeze, count_trainable_parameters, accuracy

torch.manual_seed(0)


## Partie 1 — Un modèle pré-entraîné, sans entraînement

Le plus simple pour réutiliser un modèle de vision : le `pipeline` de `transformers`. Ici un
`ResNet-50` entraîné sur ImageNet (1000 classes) — le même type d'architecture que celle
étudiée en séance 8, mais entraîné sur bien plus de données que ce que l'on peut faire en
TP. On ne fait **aucun** entraînement, juste de l'inférence, sur quelques images du dossier
`cats_and_dogs` déjà utilisé en séance 9... pardon, réservé pour la Partie 3 de cette séance.


In [ ]:
classifier = pipeline("image-classification", model="microsoft/resnet-50")

sample_paths = sorted(glob.glob("./cats_and_dogs/test_catdog/cat.0.jpg")) + \
               sorted(glob.glob("./cats_and_dogs/test_catdog/dog.0.jpg"))

fig, axes = plt.subplots(1, len(sample_paths), figsize=(4 * len(sample_paths), 4))
for ax, path in zip(axes, sample_paths):
    img = Image.open(path).convert("RGB")
    preds = classifier(img)
    ax.imshow(img)
    ax.axis("off")
    title = "\n".join(f"{p['label']} ({p['score']:.2f})" for p in preds[:3])
    ax.set_title(title, fontsize=9)
plt.tight_layout()
plt.show()


**Questions.**

- ImageNet ne contient pas de classe générique "chat" ou "chien", mais des races précises
  (`tabby cat`, `Labrador retriever`...). Le modèle s'en sort-il malgré tout pour distinguer
  chat et chien sur vos exemples ?
- Le `pipeline` renvoie un score de confiance par classe : que représente-t-il précisément
  (indice : la couche de sortie du modèle est une `softmax`) ?
- Essayez une image qui n'a clairement rien à voir avec les classes ImageNet (un objet du
  quotidien insolite, une capture d'écran...). Que se passe-t-il ?

## Partie 2 — Sous le capot : `AutoImageProcessor` + `AutoModel`

Un `pipeline` masque deux objets : un **processor** (image → tenseur normalisé, de la bonne
taille) et un **modèle** (tenseur → représentation, puis logits si le modèle a une tête de
classification). Regardons-les séparément, avec `AutoModel` (le backbone **sans** tête de
classification, contrairement à `AutoModelForImageClassification` utilisé implicitement par
le `pipeline` ci-dessus).


In [ ]:
processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
backbone = AutoModel.from_pretrained("microsoft/resnet-50")

img = Image.open(sample_paths[0]).convert("RGB")
inputs = processor(images=img, return_tensors="pt")
print("Clés produites par le processor :", list(inputs.keys()))
print("Shape de pixel_values :", inputs["pixel_values"].shape)

with torch.no_grad():
    output = backbone(**inputs)

print("last_hidden_state :", output.last_hidden_state.shape)  # carte de features avant pooling
print("pooler_output      :", output.pooler_output.shape)      # vecteur global de l'image


`microsoft/resnet-50` sait produire une représentation vectorielle d'une image
(`pooler_output`, de dimension `backbone.config.hidden_sizes[-1]`), mais ne sait pas encore
classer un chat ou un chien : il n'a pas de tête de classification pour cette tâche
spécifique. C'est exactement la même logique de transfer learning qu'en séance 9, appliquée
ici à un backbone pré-entraîné plutôt qu'à un CNN entraîné from scratch.

## Partie 3 — Backbone gelé + tête entraînée

On charge le dataset `cats_and_dogs` (local, déjà utilisé par le TP CNN historique de ce
cours) : deux dossiers `train_catdog/` et `test_catdog/`, chacun contenant des fichiers
nommés `cat.<id>.jpg` / `dog.<id>.jpg` **directement à plat** (pas de sous-dossier par
classe) — `torchvision.datasets.ImageFolder` ne convient donc pas tel quel, il faut un
petit `Dataset` maison.


In [ ]:
class CatsAndDogsDataset(Dataset):
    """Fichiers `cat.<id>.jpg` / `dog.<id>.jpg` à plat dans `folder` (label déduit du
    nom de fichier). `transform` doit renvoyer un tenseur `pixel_values` déjà prétraité
    (on réutilise directement le `processor` Hugging Face comme transform)."""

    def __init__(self, folder, transform):
        self.paths = sorted(glob.glob(f"{folder}/*.jpg"))
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = 0 if "cat" in path.split("/")[-1] else 1
        img = Image.open(path).convert("RGB")
        pixel_values = self.transform(img)
        return pixel_values, label


def hf_transform(img):
    # Renvoie directement un tenseur (C, H, W) déjà normalisé/redimensionné comme attendu
    # par le backbone -- le Trainer le traite ensuite comme une image "classique" (x, y).
    return processor(images=img, return_tensors="pt")["pixel_values"][0]


train_data = CatsAndDogsDataset("./cats_and_dogs/train_catdog", transform=hf_transform)
test_data = CatsAndDogsDataset("./cats_and_dogs/test_catdog", transform=hf_transform)

# Note : le dataset ne fournit qu'un train et un test (pas de val dédiée) -- on réutilise le
# test set comme validation ici, par souci de simplicité pour ce TP. Dans un vrai projet, on
# préférerait un découpage train/val/test à trois blocs distincts.
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(test_data, batch_size=32)

print(f"Train : {len(train_data)} images - Val/test : {len(test_data)} images")


In [ ]:
class ImageClassifier(nn.Module):
    """Backbone pré-entraîné + une tête linéaire pour la classification binaire chat/chien."""

    def __init__(self, backbone, n_classes=2):
        super().__init__()
        self.backbone = backbone
        hidden_size = backbone.config.hidden_sizes[-1]
        self.head = nn.Linear(hidden_size, n_classes)

    def forward(self, pixel_values):
        out = self.backbone(pixel_values=pixel_values)
        pooled = out.pooler_output.flatten(1)  # (batch, hidden_size, 1, 1) -> (batch, hidden_size)
        return self.head(pooled)


model = ImageClassifier(backbone)

# On gèle tout le backbone : seule la tête (quelques milliers de paramètres) est entraînée.
freeze(model.backbone)
print("Paramètres entraînables (backbone gelé) :", count_trainable_parameters(model))

optimizer = torch.optim.Adam(model.head.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

trainer = Trainer(model, optimizer, loss_fn, metrics={"acc": accuracy})
history_frozen = trainer.fit(train_loader, val_loader, epochs=3)


## Partie 4 — Dégeler quelques couches

Le backbone gelé sert de simple extracteur de features génériques (apprises sur ImageNet).
En dégelant les dernières couches (celles qui encodent les notions les plus spécifiques, par
opposition aux premières couches qui détectent des motifs très génériques comme des
contours), on peut souvent gagner en performance, au prix d'un entraînement plus coûteux.

**À vous de jouer.** `backbone.encoder.stages` est la liste des "étages" du ResNet
(inspectez sa longueur et sa structure avec `print(backbone.encoder.stages)` si besoin).
Dégelez le **dernier** étage uniquement.


In [ ]:
print(f"Nombre d'étages du backbone : {len(model.backbone.encoder.stages)}")

# TODO : dégeler le dernier étage du backbone (indice : unfreeze(...), cf. training_toolbox.py)
# unfreeze(...)

print("Paramètres entraînables après dégel partiel :", count_trainable_parameters(model))

# Learning rate plus faible : on ne veut pas détruire ce que le backbone a appris sur ImageNet.
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad], lr=1e-5
)

trainer = Trainer(model, optimizer, loss_fn, metrics={"acc": accuracy})
history_finetuned = trainer.fit(train_loader, val_loader, epochs=2)


**Questions.**

- Comparez `history_frozen` et `history_finetuned` (accuracy de validation). Le gain observé
  justifie-t-il le coût de calcul supplémentaire (davantage de paramètres à mettre à jour,
  learning rate plus prudent) ?
- Comparez cette approche à celle du mini-projet CNN from scratch de la séance 9 : combien
  d'images et d'epochs fallait-il là-bas pour obtenir une bonne accuracy, contre combien ici ?
  D'où vient la différence ?
- Que se passerait-il, à votre avis, si on dégelait *tout* le backbone d'un coup avec un jeu
  d'entraînement aussi petit (quelques centaines d'images) ?

## Partie 5 — Réutiliser un modèle génératif pré-entraîné

Retour au générateur, sans repasser par l'entraînement adversarial de la séance 10 : on
charge un modèle de **diffusion** déjà entraîné (famille de modèles génératifs différente des
GAN, egalement étudiée en cours), et on l'utilise uniquement en inférence — génération
"gratuite" grâce à la réutilisation. On choisit un modèle volontairement petit
(`google/ddpm-cifar10-32`, images 32x32) pour rester utilisable sur CPU en séance ; le nombre
de pas de débruitage (`num_inference_steps`) est réduit par rapport à la valeur par défaut
(1000) pour limiter le temps de génération.


In [ ]:
ddpm = DDPMPipeline.from_pretrained("google/ddpm-cifar10-32")
# ddpm.to("cuda")  # si vous disposez d'un GPU

images = ddpm(batch_size=4, num_inference_steps=200).images

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, img in zip(axes, images):
    ax.imshow(img)
    ax.axis("off")
plt.suptitle("Échantillons générés par un DDPM pré-entraîné (CIFAR-10)")
plt.tight_layout()
plt.show()


**Questions.**

- Un GAN (séance 10) génère une image en un seul passage dans le générateur. Un modèle de
  diffusion, lui, itère `num_inference_steps` fois. Quel est l'impact sur le temps de
  génération ? Qu'est-ce que ça change en termes de stabilité d'entraînement (pas besoin de
  compétition générateur/discriminateur ici) ?
- Essayez de réduire `num_inference_steps` (par exemple à 20). Que se passe-t-il sur la
  qualité des images générées ?
- Comparez visuellement ces échantillons à ceux obtenus avec le GAN de la séance 10 sur
  Fashion-MNIST. Les deux ne sont pas entraînés sur les mêmes données : la comparaison porte
  sur la *diversité* et la *netteté* des échantillons, pas sur le contenu.

## Pour aller plus loin (optionnel, coûteux en calcul)

- `StableDiffusionPipeline` (modèle `runwayml/stable-diffusion-v1-5`) permet de générer une
  image à partir d'un prompt texte : le principe est le même que ci-dessus, mais le
  débruitage est *conditionné* par une représentation du texte. Beaucoup plus lourd (à
  réserver à un GPU, hors séance) :

```python
from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16
)
pipe = pipe.to("cuda")
image = pipe("a photo of a computer science teacher surfing a gigantic wave").images[0]
```

- Remplacer `microsoft/resnet-50` par un ViT (`google/vit-base-patch16-224`) en Partie 1-3 :
  même démarche, architecture différente (transformer plutôt que convolutif) — l'occasion de
  vérifier que l'API Hugging Face reste identique d'un type d'architecture à l'autre.
- Utiliser `EarlyStopping` / `ModelCheckpoint` (déjà dans `training_toolbox.py`) pendant le
  fine-tuning de la Partie 3-4.
